# Skew Detection Algorithm Visualization

This notebook visualizes all steps of the MCCSD (Modified Cross-Correlation Skew Detection) algorithm.

## Steps:
1. Load image and run layout analysis
2. Filter to text regions only
3. Calculate cross-correlations (VCC and HCC)
4. Find peaks in correlation functions
5. Detect skew angle
6. Apply correction
7. Compare before/after

In [ ]:
import sys
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from ocr_reflow.layout import layout
from ocr_reflow.skew_detection import (
    calculate_vertical_cross_correlation,
    calculate_horizontal_cross_correlation,
    calculate_total_variation,
    find_peaks,
    detect_skew_in_region,
    detect_skew_in_text_regions,
    rotate_image
)

# Set up matplotlib
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 10

print("✓ Imports successful")

## 1. Load Image and Run Layout Analysis

In [ ]:
# Choose an image to analyze
image_path = '../images/dvurog_p017.png'  # Change this to test different images
# Other examples:
# image_path = '../images/sedg_p598.png'
# image_path = '../images/dvurog_p021.png'

print(f"Loading image: {image_path}")
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

print(f"Image size: {img.shape[1]}x{img.shape[0]}")

# Display original image
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Run layout analysis
print("Running layout analysis...")
layout_boxes = layout(image_path)
print(f"\nFound {len(layout_boxes)} layout boxes:")

# Categorize boxes
text_boxes = []
other_boxes = []

for geom, box_type in layout_boxes:
    bounds = geom.bounds
    print(f"  {box_type:20s}: ({bounds[0]:6.0f}, {bounds[1]:6.0f}, {bounds[2]:6.0f}, {bounds[3]:6.0f})")
    
    if box_type in ["plain text", "title"]:
        text_boxes.append((geom, box_type))
    else:
        other_boxes.append((geom, box_type))

print(f"\nText boxes: {len(text_boxes)}")
print(f"Other boxes: {len(other_boxes)}")

## 2. Visualize Layout Boxes

In [ ]:
# Visualize layout with color coding
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
ax.imshow(img_rgb)

# Color scheme
colors = {
    "plain text": 'lime',
    "title": 'cyan',
    "figure": 'red',
    "figure_caption": 'orange',
    "figure_and_caption": 'red',
    "table": 'magenta',
    "table_caption": 'pink',
    "table_and_caption": 'magenta',
    "isolate_formula": 'yellow',
    "formula_caption": 'gold',
    "isolate_formula_and_caption": 'yellow',
    "abandon": 'gray'
}

for geom, box_type in layout_boxes:
    bounds = geom.bounds
    x, y, x2, y2 = bounds
    width = x2 - x
    height = y2 - y
    
    color = colors.get(box_type, 'white')
    linewidth = 3 if box_type in ["plain text", "title"] else 2
    
    rect = patches.Rectangle((x, y), width, height, linewidth=linewidth,
                             edgecolor=color, facecolor='none', label=box_type)
    ax.add_patch(rect)
    
    # Add label
    ax.text(x, y - 10, box_type, color=color, fontsize=10, weight='bold',
           bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))

ax.set_title('Layout Analysis: Green = Text Regions (used for skew detection)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\n✓ Text regions (green) will be used for skew detection")
print(f"✓ Other regions (red/yellow/etc.) will be IGNORED")

## 3. Extract and Visualize Sample Text Regions

In [ ]:
# Extract sample regions from text boxes
if text_boxes:
    num_samples = min(3, len(text_boxes))
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    if num_samples == 1:
        axes = [axes]
    
    for i, (geom, box_type) in enumerate(text_boxes[:num_samples]):
        bounds = geom.bounds
        x1, y1, x2, y2 = int(bounds[0]), int(bounds[1]), int(bounds[2]), int(bounds[3])
        
        # Extract region
        region = img_rgb[y1:y2, x1:x2]
        
        axes[i].imshow(region)
        axes[i].set_title(f'Text Box {i+1}: {box_type}\n{region.shape[1]}×{region.shape[0]}px')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No text boxes found!")

## 4. Calculate Cross-Correlations for a Sample Region

In [ ]:
# Use the first text box for detailed analysis
if text_boxes:
    geom, box_type = text_boxes[0]
    bounds = geom.bounds
    x1, y1, x2, y2 = int(bounds[0]), int(bounds[1]), int(bounds[2]), int(bounds[3])
    
    # Extract region (grayscale)
    test_region = gray[y1:y2, x1:x2]
    
    print(f"Analyzing region from first text box:")
    print(f"  Position: ({x1}, {y1}) to ({x2}, {y2})")
    print(f"  Size: {test_region.shape[1]}×{test_region.shape[0]}px")
    
    # Calculate correlations
    d = 75
    s_range = 25
    
    print(f"\nCalculating cross-correlations with d={d}, s_range={s_range}...")
    R_V = calculate_vertical_cross_correlation(test_region, d, s_range)
    R_H = calculate_horizontal_cross_correlation(test_region, d, s_range)
    
    delta_V = calculate_total_variation(R_V)
    delta_H = calculate_total_variation(R_H)
    
    print(f"\nResults:")
    print(f"  VCC total variation: {delta_V:.2e}")
    print(f"  HCC total variation: {delta_H:.2e}")
    print(f"  Selected: {'VCC (horizontal text)' if delta_V > delta_H else 'HCC (vertical text)'}")
else:
    print("No text boxes to analyze!")

## 5. Visualize Cross-Correlation Functions

In [ ]:
if text_boxes:
    # Plot correlation functions
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    s_values = np.arange(-s_range, s_range + 1)
    
    # VCC plot
    axes[0, 0].plot(s_values, R_V, 'b-', linewidth=2)
    axes[0, 0].scatter([s_values[np.argmax(R_V)]], [R_V.max()], color='red', s=100, zorder=5, label='Peak')
    axes[0, 0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[0, 0].set_xlabel('Shift s (pixels)')
    axes[0, 0].set_ylabel('Correlation R(s)')
    axes[0, 0].set_title(f'Vertical Cross-Correlation (VCC)\nTotal Variation: {delta_V:.2e}')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    
    # HCC plot
    axes[0, 1].plot(s_values, R_H, 'g-', linewidth=2)
    axes[0, 1].scatter([s_values[np.argmax(R_H)]], [R_H.max()], color='red', s=100, zorder=5, label='Peak')
    axes[0, 1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[0, 1].set_xlabel('Shift s (pixels)')
    axes[0, 1].set_ylabel('Correlation R(s)')
    axes[0, 1].set_title(f'Horizontal Cross-Correlation (HCC)\nTotal Variation: {delta_H:.2e}')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()
    
    # Selected correlation (zoomed)
    R_selected = R_V if delta_V > delta_H else R_H
    peaks = find_peaks(R_selected, s_range)
    
    axes[1, 0].plot(s_values, R_selected, 'r-', linewidth=2, label='Selected correlation')
    for s_val, peak_val in peaks[:5]:  # Show top 5 peaks
        idx = s_val + s_range
        axes[1, 0].scatter([s_val], [peak_val], s=100, zorder=5)
        axes[1, 0].annotate(f's={s_val}', (s_val, peak_val), 
                          xytext=(5, 5), textcoords='offset points')
    
    axes[1, 0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 0].set_xlabel('Shift s (pixels)')
    axes[1, 0].set_ylabel('Correlation R(s)')
    axes[1, 0].set_title('Selected Correlation with Detected Peaks')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    
    # Calculate angle from primary peak
    if peaks:
        s_p, _ = peaks[0]
        angle = np.arctan(s_p / d) * 180.0 / np.pi
        
        # Show angle calculation
        axes[1, 1].text(0.5, 0.7, f'Primary Peak:', ha='center', fontsize=14, weight='bold', 
                       transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.5, 0.55, f's = {s_p}', ha='center', fontsize=16, 
                       transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.5, 0.4, f'd = {d}', ha='center', fontsize=16, 
                       transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.5, 0.25, f'Angle = arctan(s/d)', ha='center', fontsize=14, 
                       transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.5, 0.1, f'= arctan({s_p}/{d})', ha='center', fontsize=14, 
                       transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.5, 0.5, f'= {angle:.2f}°', ha='center', fontsize=20, weight='bold',
                       color='red', transform=axes[1, 1].transAxes,
                       bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.5))
        axes[1, 1].axis('off')
        axes[1, 1].set_title('Calculated Skew Angle', fontsize=14, weight='bold')
    
    plt.tight_layout()
    plt.show()
    
    if peaks:
        print(f"\n✓ Detected skew in this region: {angle:.2f}°")

## 6. Full Skew Detection (All Text Regions)

In [ ]:
# Run full skew detection on all text regions
print("Running skew detection on all text regions...")
detected_angle = detect_skew_in_text_regions(img, layout_boxes)

print(f"\n" + "="*70)
print(f"FINAL DETECTED SKEW ANGLE: {detected_angle:.2f}°")
print("="*70)

## 7. Apply Skew Correction

In [ ]:
# Apply correction
if abs(detected_angle) > 0.1:
    print(f"Applying rotation of {detected_angle:.2f}°...")
    corrected_img = rotate_image(img, detected_angle)
    corrected_rgb = cv2.cvtColor(corrected_img, cv2.COLOR_BGR2RGB)
    
    # Display side-by-side comparison
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    axes[0].imshow(img_rgb)
    axes[0].set_title(f'Original Image\nDetected Skew: {detected_angle:.2f}°', fontsize=14, weight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(corrected_rgb)
    axes[1].set_title(f'Corrected Image\nRotated by: {detected_angle:.2f}°', fontsize=14, weight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Correction applied!")
    print(f"  Original size: {img.shape[1]}×{img.shape[0]}")
    print(f"  Corrected size: {corrected_img.shape[1]}×{corrected_img.shape[0]}")
else:
    print(f"\nSkew angle ({detected_angle:.2f}°) too small, no correction needed.")
    corrected_img = img
    corrected_rgb = img_rgb

## 8. Detailed Region Analysis (Optional)

In [ ]:
# Analyze multiple regions and show angle distribution
print("Analyzing multiple random regions from text boxes...")

region_angles = []
region_size = 150
num_samples = 15

for i in range(num_samples):
    if text_boxes:
        # Randomly select a text box
        geom, _ = text_boxes[np.random.randint(len(text_boxes))]
        bounds = geom.bounds
        x1, y1, x2, y2 = int(bounds[0]), int(bounds[1]), int(bounds[2]), int(bounds[3])
        
        box_width = x2 - x1
        box_height = y2 - y1
        
        if box_width >= region_size and box_height >= region_size:
            # Random region within box
            rx = np.random.randint(x1, x2 - region_size + 1)
            ry = np.random.randint(y1, y2 - region_size + 1)
            region = gray[ry:ry+region_size, rx:rx+region_size]
        else:
            # Use whole box
            region = gray[y1:y2, x1:x2]
        
        angle = detect_skew_in_region(region, d=75, s_range=25)
        if angle is not None:
            region_angles.append(angle)

if region_angles:
    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(region_angles, bins=20, edgecolor='black', alpha=0.7)
    axes[0].axvline(x=np.median(region_angles), color='red', linestyle='--', 
                   linewidth=2, label=f'Median: {np.median(region_angles):.2f}°')
    axes[0].axvline(x=np.mean(region_angles), color='green', linestyle='--', 
                   linewidth=2, label=f'Mean: {np.mean(region_angles):.2f}°')
    axes[0].set_xlabel('Detected Angle (degrees)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title(f'Distribution of Detected Angles\n({len(region_angles)} regions)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot(region_angles, vert=True)
    axes[1].set_ylabel('Detected Angle (degrees)')
    axes[1].set_title('Angle Distribution (Box Plot)')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"\nStatistics from {len(region_angles)} regions:")
    print(f"  Mean: {np.mean(region_angles):.2f}°")
    print(f"  Median: {np.median(region_angles):.2f}°")
    print(f"  Std Dev: {np.std(region_angles):.2f}°")
    print(f"  Min: {np.min(region_angles):.2f}°")
    print(f"  Max: {np.max(region_angles):.2f}°")
    print(f"  Range: {np.max(region_angles) - np.min(region_angles):.2f}°")
else:
    print("No valid regions found for analysis.")

## 9. Compare with Full-Image Detection

In [ ]:
# Compare text-region vs full-image detection
from ocr_reflow.skew_detection import detect_skew

print("Comparing detection methods...\n")

# Full image detection
full_image_angle = detect_skew(img)

# Text region detection (already computed)
text_region_angle = detected_angle

# Display comparison
print("="*70)
print(f"COMPARISON:")
print("="*70)
print(f"Full-image detection:    {full_image_angle:6.2f}°")
print(f"Text-region detection:   {text_region_angle:6.2f}°")
print(f"Difference:              {abs(full_image_angle - text_region_angle):6.2f}°")
print("="*70)

if abs(full_image_angle - text_region_angle) > 1.0:
    print("\n⚠️  Significant difference detected!")
    print("   Text-region method is more accurate (ignores figures/formulas)")
else:
    print("\n✓ Both methods agree (difference < 1°)")

## 10. Save Corrected Image (Optional)

In [ ]:
# Save corrected image
if abs(detected_angle) > 0.1:
    output_path = image_path.replace('.png', '_corrected.png')
    cv2.imwrite(output_path, corrected_img)
    print(f"✓ Corrected image saved to: {output_path}")
else:
    print("No correction needed, skipping save.")

## Summary

This notebook demonstrates the complete skew detection pipeline:

1. **Layout Analysis**: Identifies text regions vs figures/formulas
2. **Text-Only Detection**: Analyzes only plain text and title boxes
3. **Cross-Correlation**: Calculates VCC and HCC to find text line patterns
4. **Peak Detection**: Identifies the shift that maximizes correlation
5. **Angle Calculation**: Converts peak shift to rotation angle
6. **Validation**: Checks consistency and avoids false positives
7. **Correction**: Applies rotation to deskew the document

### Key Advantages of Text-Region Method:
- ✅ Ignores figures, formulas, and tables
- ✅ More accurate on mixed-content pages
- ✅ Conservative validation prevents false positives
- ✅ Works well with minimal text content